Features:
- distance_from_home: continua
- distance_from_last_transaction: continua
- ratio_to_median_purchase_price: continua
- repeat_retailer: binaria
- used_chip: binaria
- used_pin_number: binaria
- online_order: binaria

Target:
- fraud: binaria

# Detecção de Fraude em Transações de Cartão de Crédito
### Projeto Final — Introdução ao Aprendizado de Máquina (IML 2026)

---

## Introdução

A fraude em cartões de crédito representa um problema de alto impacto financeiro e social. Instituições bancárias processam milhões de transações diariamente e precisam identificar, em tempo real, quais operações são potencialmente fraudulentas — sem bloquear indevidamente transações legítimas.

Este projeto aplica técnicas de **Aprendizado de Máquina Supervisionado** para classificar transações como legítimas ou fraudulentas, com foco em dois métodos clássicos de análise discriminante: **LDA** (Linear Discriminant Analysis) e **QDA** (Quadratic Discriminant Analysis).

## Objetivo

Desenvolver e avaliar modelos de classificação capazes de detectar fraudes em transações de cartão de crédito, abordando o desafio do **desbalanceamento de classes** com uma estratégia de geração de dados sintéticos baseada em distribuições estatísticas.

## Dataset

O dataset utilizado é o **Credit Card Fraud** disponível publicamente no Kaggle (`dhanushnarayananr/credit-card-fraud`), contendo **1.000.000 transações** com 7 atributos preditores e 1 variável alvo.

### Descrição das Features

| Feature | Tipo | Descrição |
|---------|------|-----------|
| `distance_from_home` | Contínua | Distância entre o local da transação e a residência do cliente (km) |
| `distance_from_last_transaction` | Contínua | Distância entre a transação atual e a anterior (km) |
| `ratio_to_median_purchase_price` | Contínua | Razão entre o valor da compra e o preço mediano histórico do cliente |
| `repeat_retailer` | Binária | 1 se o cliente já comprou neste estabelecimento antes |
| `used_chip` | Binária | 1 se a transação usou o chip físico do cartão |
| `used_pin_number` | Binária | 1 se a senha PIN foi utilizada |
| `online_order` | Binária | 1 se a transação foi realizada online |

**Variável alvo:**
- `fraud`: Binária — 0 para transação legítima, 1 para transação fraudulenta

---
## 1. Importações e Configurações

Importamos as bibliotecas necessárias para manipulação de dados e definimos as constantes globais que organizam as features por tipo. Manter a ordem das colunas como constante evita erros de indexação ao longo do pipeline.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# MANTER ORDEM DAS COLUNAS PARA EVITAR ERROS DE INDEXAÇÃO
COLUNAS_CONTINUAS = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price"
]

COLUNAS_BINARIAS = [
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

---
## 2. Pré-processamento

### 2.1 Estratégia de Balanceamento — Oversampling Paramétrico

O dataset apresenta um **desbalanceamento significativo**: a grande maioria das transações é legítima, enquanto as fraudes representam uma minoria. Treinar um classificador diretamente com essa distribuição levaria a um modelo enviesado, capaz de atingir alta acurácia simplesmente "chutando" sempre a classe majoritária, sem aprender a identificar fraudes.

A estratégia adotada é o **oversampling paramétrico** da classe minoritária (fraudes) no conjunto de treino. Em vez de duplicar amostras reais ou interpolar entre vizinhos, **aprendemos a distribuição estatística** das fraudes e geramos novas amostras a partir dela:

- **Features contínuas**: modeladas por uma **Distribuição Gaussiana Multivariada** $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, estimando a média $\boldsymbol{\mu}$ e a matriz de covariância $\boldsymbol{\Sigma}$ das fraudes reais. Isso captura não só a variância individual de cada feature, mas também as **correlações entre elas**.

- **Features binárias**: modeladas por **Distribuições de Bernoulli** independentes, uma por feature, com parâmetro $p$ estimado como a proporção de 1s entre as fraudes reais.

> **Importante**: o oversampling é aplicado **exclusivamente no conjunto de treino**, após o split treino/teste. O conjunto de teste permanece intacto com dados 100% reais, garantindo que as métricas de avaliação reflitam o desempenho no mundo real.

In [ ]:
def gen_continuas(df_train_fraud, n_sintetico):
    """
    Gera dados sintéticos contínuos usando distribuição multivariada normal
    """
    X_cont = df_train_fraud[COLUNAS_CONTINUAS].values
    mu = X_cont.mean(axis=0)
    Sigma = np.cov(X_cont, rowvar=False)
    rng = np.random.default_rng(seed=4267)
    X_cont_sintetico = rng.multivariate_normal(mu, Sigma, size=n_sintetico)
    return X_cont_sintetico

def gen_binarias(df_train_fraud, n_sintetico):
    """
    Gera dados sintéticos binários usando Bernoulli

    """
    prob_binarias = {}
    rng = np.random.default_rng(seed=4267)

    for col in COLUNAS_BINARIAS:
        prob_binarias[col] = df_train_fraud[col].mean()
    
    binarias_sinteticas = {}
    for col in COLUNAS_BINARIAS:
        p = prob_binarias[col]
        binarias_sinteticas[col] = rng.binomial(1, p, size=n_sintetico)
    
    return binarias_sinteticas

### 2.2 Carregamento e Divisão Treino/Teste

O dataset é dividido em **80% treino** e **20% teste**. O parâmetro `stratify=y` garante que a proporção de fraudes seja preservada em ambos os conjuntos — fundamental em datasets desbalanceados. A semente `random_state=4267` assegura reprodutibilidade dos resultados.

In [ ]:
df = pd.read_csv("dataset/card_transdata.csv")

In [ ]:
# Separar train e teste para depois gerar os dados sintéticos apenas no conjunto de treino
from sklearn.model_selection import train_test_split
# Drop da coluna target
X = df[COLUNAS_CONTINUAS + COLUNAS_BINARIAS].values
y = df["fraud"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4267, stratify=y)

### 2.3 Geração dos Dados Sintéticos e Balanceamento

A quantidade de amostras sintéticas é calculada como:

$$n_{\text{sintético}} = \left\lfloor \frac{|n_{\text{legítimas}} - n_{\text{fraudes}}|}{1.5} \right\rfloor$$

O divisor **1.5** é uma escolha deliberada: em vez de equalizar completamente as classes (50/50), reduzimos o desbalanceamento de forma moderada. Oversampling excessivo pode levar o modelo a aprender padrões artificiais dos dados sintéticos, prejudicando a generalização.

O dataset de treino balanceado é formado pela **concatenação** do treino original com os dados sintéticos gerados. A distribuição resultante é exibida abaixo:

In [88]:
df_train["fraud"] = y_train

df_balanceado_train = pd.concat(
    [df_train, df_sintetico],
    ignore_index=True
)

print(df_balanceado_train["fraud"].value_counts())

fraud
0.0    730078
1.0    510026
Name: count, dtype: int64


Após o balanceamento, o treino passou de uma proporção de ~91%/9% para aproximadamente **59%/41%** — redução substancial do desbalanceamento sem eliminá-lo completamente.

In [89]:
X_train_balanceado = df_balanceado_train.drop(columns=["fraud"]).values
y_train_balanceado = df_balanceado_train["fraud"].values

---
## 3. Modelagem

Treinamos dois classificadores da família de **Análise Discriminante**, ambos baseados na hipótese de que cada classe segue uma distribuição Gaussiana multivariada.

### 3.1 LDA — Linear Discriminant Analysis

In [90]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis        
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_balanceado, y_train_balanceado)
y_pred = lda.predict(X_test)

In [91]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {acc}")
print(f"Precision: {prec}")
print(f"Recall: {rec}")
print(f"F1-Score: {f1}")


Accuracy: 0.958935
Precision: 0.7358269720101781
Recall: 0.8271265945884103
F1-Score: 0.778810158627562


### 3.2 QDA — Quadratic Discriminant Analysis

In [92]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis        

qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_balanceado, y_train_balanceado)
y_pred = qda.predict(X_test)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {acc}")
print(f"Precision: {prec}")
print(f"Recall: {rec}")
print(f"F1-Score: {f1}")


Accuracy: 0.942525
Precision: 0.6082224472085623
Recall: 0.9622447228419427
F1-Score: 0.7453308815384275


---
## 4. Análise Comparativa dos Resultados

EXPLICAR NO FINAL